# 09 — Natural Language Processing (NLP)

## Leadership and Management Book Recommendation System

### Objective

This notebook transforms the textual metadata of the integrated leadership and management book catalogue into numerical representations suitable for similarity analysis and content-based recommendation.

The feature-engineering stage identified substantial differences in textual metadata richness between the two data sources.

The catalogue therefore contains two prepared textual representations:

### Core Content

- title
- author

This representation provides the most comparable textual basis across the catalogue.

### Enriched Content

- title
- author
- subjects
- description

This representation uses all available semantic metadata but contains greater variation in information richness because subjects and descriptions are primarily available through Open Library.

## NLP Strategy

The analysis will:

1. validate the engineered dataset and prepared text fields
2. perform transparent text normalization
3. examine vocabulary characteristics
4. construct TF-IDF representations
5. compare core and enriched representations
6. evaluate vocabulary size and matrix sparsity
7. inspect representative TF-IDF terms
8. assess source-related differences in textual representation
9. save reproducible NLP artifacts for downstream modelling

TF-IDF is used because the project requires interpretable content-based representations of books while reducing the influence of terms that occur frequently throughout the catalogue.

The NLP stage does not yet produce final recommendations. Similarity computation and recommendation logic will be developed in the subsequent recommendation-system stage.

In [5]:
# ============================================================
# IMPORT LIBRARIES AND DEFINE PATHS
# ============================================================

from pathlib import Path
import re
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

FEATURES_PATH = (
    DATA_PROCESSED
    / "books_features.csv"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("NLP ENVIRONMENT")
print("=" * 75)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature file: {FEATURES_PATH}")
print(f"Models dir:   {MODELS_DIR}")

print("\nFeature file exists:", FEATURES_PATH.exists())
print("Models directory exists:", MODELS_DIR.exists())

NLP ENVIRONMENT
Project root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Feature file: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/books_features.csv
Models dir:   /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models

Feature file exists: True
Models directory exists: True


In [2]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 6.3 MB/s  0:00:01m 6.3 MB/s eta 0:00:01
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sklearn

print("scikit-learn version:", sklearn.__version__)
print("scikit-learn location:", sklearn.__file__)

scikit-learn version: 1.9.1
scikit-learn location: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/.venv/lib/python3.13/site-packages/sklearn/__init__.py


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("✓ TfidfVectorizer imported successfully")

✓ TfidfVectorizer imported successfully


## 1. Load and Validate the Engineered Dataset

The NLP workflow begins from the feature-engineered dataset produced in Notebook 08.

Before applying text preprocessing or vectorization, the dataset is validated to confirm:

- 2,067 canonical books are present
- `book_id` remains unique
- core and enriched text representations are available
- source classifications are preserved
- content-depth metadata are available

This ensures that NLP transformations begin from the validated book-level dataset without altering the canonical catalogue structure.

In [6]:
# ============================================================
# LOAD FEATURE-ENGINEERED DATASET
# ============================================================

books = pd.read_csv(
    FEATURES_PATH,
    low_memory=False
)

print("NLP INPUT DATASET")
print("=" * 75)

print(f"Rows:     {books.shape[0]:,}")
print(f"Columns:  {books.shape[1]:,}")
print(f"Book IDs: {books['book_id'].nunique():,}")


# ------------------------------------------------------------
# Required NLP columns
# ------------------------------------------------------------

required_nlp_columns = [
    "book_id",
    "canonical_title",
    "authors_text",
    "subjects_text",
    "description_text",
    "core_content_text",
    "enriched_content_text",
    "core_word_count",
    "enriched_word_count",
    "content_component_count",
    "content_depth",
    "source_group"
]

missing_columns = [
    col
    for col in required_nlp_columns
    if col not in books.columns
]


print("\nNLP COLUMN VALIDATION")
print("=" * 75)

for col in required_nlp_columns:
    print(
        f"{col:<30} "
        f"{'✓' if col in books.columns else 'MISSING'}"
    )


# ------------------------------------------------------------
# Integrity assertions
# ------------------------------------------------------------

assert len(books) == 2067
assert books["book_id"].nunique() == 2067
assert books["book_id"].duplicated().sum() == 0
assert len(missing_columns) == 0

assert books["core_content_text"].notna().all()
assert books["enriched_content_text"].notna().all()


print("\n✓ NLP INPUT DATASET VALIDATED")

NLP INPUT DATASET
Rows:     2,067
Columns:  69
Book IDs: 2,067

NLP COLUMN VALIDATION
book_id                        ✓
canonical_title                ✓
authors_text                   ✓
subjects_text                  ✓
description_text               ✓
core_content_text              ✓
enriched_content_text          ✓
core_word_count                ✓
enriched_word_count            ✓
content_component_count        ✓
content_depth                  ✓
source_group                   ✓

✓ NLP INPUT DATASET VALIDATED


In [7]:
# ============================================================
# NLP TEXT REPRESENTATION AUDIT
# ============================================================

text_audit = pd.DataFrame({
    "representation": [
        "Core",
        "Enriched"
    ],

    "documents": [
        books["core_content_text"].notna().sum(),
        books["enriched_content_text"].notna().sum()
    ],

    "median_words": [
        books["core_word_count"].median(),
        books["enriched_word_count"].median()
    ],

    "mean_words": [
        books["core_word_count"].mean(),
        books["enriched_word_count"].mean()
    ],

    "max_words": [
        books["core_word_count"].max(),
        books["enriched_word_count"].max()
    ]
})


print("NLP TEXT REPRESENTATION AUDIT")
print("=" * 75)

display(
    text_audit.style.format({
        "median_words": "{:.1f}",
        "mean_words": "{:.2f}",
        "max_words": "{:.0f}"
    })
)


print("\nSOURCE × CONTENT DEPTH")
print("=" * 75)

display(
    pd.crosstab(
        books["source_group"],
        books["content_depth"]
    ).reindex(
        columns=[
            "Minimal",
            "Basic",
            "Enriched",
            "Rich"
        ],
        fill_value=0
    )
)

NLP TEXT REPRESENTATION AUDIT


,representation,documents,median_words,mean_words,max_words
0,Core,2067,6.0,6.81,33
1,Enriched,2067,8.0,22.79,533



SOURCE × CONTENT DEPTH


content_depth,Minimal,Basic,Enriched,Rich
source_group,,,,
Both,0,0,2,1
LeadershipNow only,0,1117,0,0
Open Library only,3,123,633,188


## 2. Raw Text and Vocabulary Inspection

Before applying NLP normalization, the prepared text is inspected directly.

This step is used to identify:

- common vocabulary
- punctuation and formatting patterns
- numerical tokens
- differences between core and enriched text
- potentially uninformative high-frequency terms
- domain-specific terms that should remain meaningful

No terms are removed at this stage.

In particular, frequent domain terms such as *leadership*, *management*, *strategy*, and *business* are not automatically treated as stop words. Although frequent, they may contain important information for distinguishing books within the leadership and management domain.

In [8]:
# ============================================================
# SAMPLE RAW NLP DOCUMENTS
# ============================================================

sample_columns = [
    "book_id",
    "canonical_title",
    "source_group",
    "content_depth",
    "core_content_text",
    "enriched_content_text"
]

# Deterministic examples from different metadata depths
sample_books = (
    books
    .sort_values(
        ["content_component_count", "book_id"]
    )
    .groupby(
        "content_depth",
        group_keys=False
    )
    .head(2)
)


print("SAMPLE NLP DOCUMENTS")
print("=" * 80)

display(
    sample_books[sample_columns]
)

SAMPLE NLP DOCUMENTS


,book_id,canonical_title,source_group,content_depth,core_content_text,enriched_content_text
141,BOOK00142,executive leadership course volume 3,Open Library only,Minimal,executive leadership course volume 3,executive leadership course volume 3
713,BOOK00714,Your Next Five Moves,Open Library only,Minimal,Your Next Five Moves,Your Next Five Moves
53,BOOK00054,Leadership Development in Balance,Open Library only,Basic,Leadership Development in Balance Bruce J. Avolio,Leadership Development in Balance Bruce J. Avolio
80,BOOK00081,Leadership Development,Open Library only,Basic,Leadership Development Manuel London,Leadership Development Manuel London
1,BOOK00002,Leadership in Organizations,Open Library only,Enriched,Leadership in Organizations Gary A. Yukl,Leadership in Organizations Gary A. Yukl Organisation Prise de décision Entscheidungsfindung Leadership Organizational sociology Organization Deci...
2,BOOK00003,Kepemimpinan =,Open Library only,Enriched,Kepemimpinan = Karjadi M.,Kepemimpinan = Karjadi M. Leadership
0,BOOK00001,Principle-Centered Leadership,Open Library only,Rich,Principle-Centered Leadership Stephen R. Covey,Principle-Centered Leadership Stephen R. Covey Leadership Psychological aspects of Success Success Psychological aspects Commerce Success in busin...
6,BOOK00007,Leadership,Open Library only,Rich,Leadership Peter G. Northouse,"Leadership Peter G. Northouse Leadership Case studies Sociology Führung Leadership--case studies Hm1261 .n67 2018 303.3/4 ""The Third Edition of th..."


In [9]:
# ============================================================
# PRELIMINARY RAW VOCABULARY AUDIT
# ============================================================

from collections import Counter


def raw_tokens(text):
    """
    Preliminary tokenization for vocabulary inspection only.

    This is NOT the final NLP preprocessing function.
    """
    return re.findall(
        r"\b[a-zA-Z][a-zA-Z'-]*\b",
        str(text).lower()
    )


core_tokens = [
    token
    for text in books["core_content_text"]
    for token in raw_tokens(text)
]

enriched_tokens = [
    token
    for text in books["enriched_content_text"]
    for token in raw_tokens(text)
]


core_counts = Counter(core_tokens)
enriched_counts = Counter(enriched_tokens)


print("RAW VOCABULARY SUMMARY")
print("=" * 80)

print(f"Core total tokens:        {len(core_tokens):,}")
print(f"Core unique tokens:       {len(core_counts):,}")
print(f"Enriched total tokens:    {len(enriched_tokens):,}")
print(f"Enriched unique tokens:   {len(enriched_counts):,}")


print("\nTOP 30 CORE TOKENS")
print("=" * 80)

display(
    pd.DataFrame(
        core_counts.most_common(30),
        columns=["token", "frequency"]
    )
)


print("\nTOP 30 ENRICHED TOKENS")
print("=" * 80)

display(
    pd.DataFrame(
        enriched_counts.most_common(30),
        columns=["token", "frequency"]
    )
)

RAW VOCABULARY SUMMARY
Core total tokens:        13,991
Core unique tokens:       4,752
Enriched total tokens:    45,709
Enriched unique tokens:   8,781

TOP 30 CORE TOKENS


,token,frequency
0,the,484
1,management,396
2,and,370
3,leadership,306
4,of,222
5,a,158
6,to,97
7,john,92
8,j,88
9,david,87



TOP 30 ENRICHED TOKENS


,token,frequency
0,the,1568
1,management,1515
2,and,1495
3,of,998
4,leadership,713
5,business,705
6,to,674
7,in,587
8,a,583
9,economics,346


## 3. Text Preprocessing Strategy

Vocabulary inspection showed that the corpus contains both common English function words and meaningful domain-specific terminology.

Frequent function words such as *the*, *and*, *of*, *to*, and *in* provide limited discriminatory value and will be handled using standard English stop-word removal.

However, frequent domain terms including *management*, *leadership*, *business*, *strategy*, *change*, and *organizational* are retained because they represent meaningful distinctions within the subject domain.

The preprocessing strategy therefore remains conservative:

- convert text to lowercase
- normalize whitespace
- remove non-semantic punctuation
- retain alphabetic and numeric information where appropriate
- exclude isolated one-character tokens during vectorization
- apply standard English stop-word removal

The analysis does not currently apply:

- stemming
- lemmatization
- translation
- manual removal of author names
- manual removal of domain-specific terminology

This preserves interpretability and avoids collapsing potentially meaningful distinctions before their effect on TF-IDF representations can be evaluated.

In [10]:
# ============================================================
# CONSERVATIVE TEXT NORMALIZATION
# ============================================================

def normalize_nlp_text(text):
    """
    Apply conservative structural normalization.

    Stop-word removal and final token filtering are handled
    later by TfidfVectorizer.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Normalize apostrophe variants
    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
    )

    # Replace punctuation with spaces while retaining
    # letters, numbers, apostrophes and hyphens
    text = re.sub(
        r"[^a-z0-9'\-\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


books["core_text_clean"] = (
    books["core_content_text"]
    .apply(normalize_nlp_text)
)

books["enriched_text_clean"] = (
    books["enriched_content_text"]
    .apply(normalize_nlp_text)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert books["core_text_clean"].ne("").all()
assert books["enriched_text_clean"].ne("").all()

assert len(books) == 2067
assert books["book_id"].nunique() == 2067


print("TEXT NORMALIZATION COMPLETE")
print("=" * 80)

print(
    f"Non-empty core documents:     "
    f"{books['core_text_clean'].ne('').sum():,}"
)

print(
    f"Non-empty enriched documents: "
    f"{books['enriched_text_clean'].ne('').sum():,}"
)

print("\n✓ ALL 2,067 BOOKS RETAINED")

TEXT NORMALIZATION COMPLETE
Non-empty core documents:     2,067
Non-empty enriched documents: 2,067

✓ ALL 2,067 BOOKS RETAINED


In [11]:
# ============================================================
# BEFORE / AFTER NORMALIZATION CHECK
# ============================================================

comparison_sample = (
    books[
        [
            "book_id",
            "canonical_title",
            "core_content_text",
            "core_text_clean",
            "enriched_content_text",
            "enriched_text_clean"
        ]
    ]
    .sample(
        n=8,
        random_state=42
    )
)


print("TEXT NORMALIZATION SAMPLE")
print("=" * 80)

display(comparison_sample)

TEXT NORMALIZATION SAMPLE


,book_id,canonical_title,core_content_text,core_text_clean,enriched_content_text,enriched_text_clean
220,BOOK00221,Transformational leadership,Transformational leadership Randy Dobbs,transformational leadership randy dobbs,Transformational leadership Randy Dobbs Transformational leadership Leadership Organizational change,transformational leadership randy dobbs transformational leadership leadership organizational change
548,BOOK00549,Contemporary Project Management,Contemporary Project Management Timothy Kloppenborg Kathryn Wells Vittal S. Anantatmula,contemporary project management timothy kloppenborg kathryn wells vittal s anantatmula,Contemporary Project Management Timothy Kloppenborg Kathryn Wells Vittal S. Anantatmula Project management,contemporary project management timothy kloppenborg kathryn wells vittal s anantatmula project management
1333,BOOK01334,The Upside of Disruption,The Upside of Disruption Terence Mauri,the upside of disruption terence mauri,The Upside of Disruption Terence Mauri,the upside of disruption terence mauri
196,BOOK00197,Transformational Leadership,Transformational Leadership Glenn Walter,transformational leadership glenn walter,Transformational Leadership Glenn Walter,transformational leadership glenn walter
29,BOOK00030,Reframing Organizations,Reframing Organizations Lee G. Bolman,reframing organizations lee g bolman,Reframing Organizations Lee G. Bolman Leadership Management Organizational behavior Business Nonfiction Comportement organisationnel Gestion Cultu...,reframing organizations lee g bolman leadership management organizational behavior business nonfiction comportement organisationnel gestion cultur...
184,BOOK00185,Team leadership in action,Team leadership in action Laura Mae Douglass,team leadership in action laura mae douglass,Team leadership in action Laura Mae Douglass Nursing services Leadership Administration Team nursing Team Nursing Nursing Services Organization & ...,team leadership in action laura mae douglass nursing services leadership administration team nursing team nursing nursing services organization ad...
942,BOOK00943,The management of innovation,The management of innovation Tom R. Burns G. M. Stalker,the management of innovation tom r burns g m stalker,The management of innovation Tom R. Burns G. M. Stalker Electronic industries Technological innovations Industrial management Great Britain Manage...,the management of innovation tom r burns g m stalker electronic industries technological innovations industrial management great britain managemen...
582,BOOK00583,Armstrong's handbook of performance management,Armstrong's handbook of performance management Michael Armstrong,armstrong's handbook of performance management michael armstrong,"Armstrong's handbook of performance management Michael Armstrong Business Nonfiction Employees, rating of Performance standards Performance Employ...",armstrong's handbook of performance management michael armstrong business nonfiction employees rating of performance standards performance employe...


## 4. Core TF-IDF Representation

The first TF-IDF model uses the Core representation consisting of title and author metadata.

The initial vectorizer uses:

- English stop-word removal
- unigrams and bigrams
- minimum document frequency of 2
- maximum document frequency of 95%
- sublinear term-frequency scaling
- exclusion of single-character tokens

Bigrams are included because multi-word concepts such as *project management*, *emotional intelligence*, *organizational behavior*, and *strategic management* may carry more specific semantic meaning than their individual words.

The Core model provides the more source-comparable baseline representation.

In [12]:
# ============================================================
# CORE TF-IDF VECTORIZATION
# ============================================================

core_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]+\b"
)


core_tfidf = core_vectorizer.fit_transform(
    books["core_text_clean"]
)


core_feature_names = (
    core_vectorizer.get_feature_names_out()
)


# ------------------------------------------------------------
# Matrix diagnostics
# ------------------------------------------------------------

core_total_cells = (
    core_tfidf.shape[0]
    * core_tfidf.shape[1]
)

core_nonzero = core_tfidf.nnz

core_density = (
    core_nonzero
    / core_total_cells
)

core_sparsity = (
    1 - core_density
)


print("CORE TF-IDF MATRIX")
print("=" * 80)

print(
    f"Documents:          "
    f"{core_tfidf.shape[0]:,}"
)

print(
    f"TF-IDF features:    "
    f"{core_tfidf.shape[1]:,}"
)

print(
    f"Non-zero values:    "
    f"{core_nonzero:,}"
)

print(
    f"Matrix density:     "
    f"{core_density:.6f}"
)

print(
    f"Matrix sparsity:    "
    f"{core_sparsity:.2%}"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert core_tfidf.shape[0] == 2067
assert core_tfidf.shape[1] > 0
assert core_tfidf.nnz > 0

print("\n✓ CORE TF-IDF REPRESENTATION CREATED")

CORE TF-IDF MATRIX
Documents:          2,067
TF-IDF features:    1,935
Non-zero values:    9,753
Matrix density:     0.002438
Matrix sparsity:    99.76%

✓ CORE TF-IDF REPRESENTATION CREATED


In [13]:
# ============================================================
# CORE TF-IDF VOCABULARY INSPECTION
# ============================================================

core_document_frequency = np.asarray(
    (core_tfidf > 0).sum(axis=0)
).ravel()


core_vocab_audit = (
    pd.DataFrame({
        "term": core_feature_names,
        "document_frequency": core_document_frequency
    })
    .sort_values(
        "document_frequency",
        ascending=False
    )
    .reset_index(drop=True)
)


print("MOST WIDESPREAD CORE TF-IDF TERMS")
print("=" * 80)

display(
    core_vocab_audit.head(30)
)


print("\nSAMPLE BIGRAMS")
print("=" * 80)

core_bigrams = core_vocab_audit[
    core_vocab_audit["term"].str.contains(" ")
]

display(
    core_bigrams.head(30)
)

MOST WIDESPREAD CORE TF-IDF TERMS


,term,document_frequency
0,management,390
1,leadership,300
2,john,85
3,david,83
4,business,69
5,human,67
6,strategic,64
7,robert,59
8,change,58
9,organizational,57



SAMPLE BIGRAMS


,term,document_frequency
16,human resource,49
18,project management,49
19,resource management,49
20,operations management,49
24,organizational behavior,48
26,performance management,46
28,emotional intelligence,45
30,strategic management,44
36,servant leadership,38
38,leadership development,38


## 5. Enriched TF-IDF Representation

A second TF-IDF representation is constructed using the Enriched content:

- title
- author
- subjects
- description

The same vectorizer configuration used for the Core representation is retained so that differences between the resulting matrices primarily reflect differences in textual content rather than different preprocessing parameters.

Because subjects and descriptions are not uniformly available across sources, the Enriched representation is treated as an additional semantic model rather than an automatic replacement for the Core representation.

In [14]:
# ============================================================
# ENRICHED TF-IDF VECTORIZATION
# ============================================================

enriched_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]+\b"
)


enriched_tfidf = enriched_vectorizer.fit_transform(
    books["enriched_text_clean"]
)


enriched_feature_names = (
    enriched_vectorizer.get_feature_names_out()
)


# ------------------------------------------------------------
# Matrix diagnostics
# ------------------------------------------------------------

enriched_total_cells = (
    enriched_tfidf.shape[0]
    * enriched_tfidf.shape[1]
)

enriched_nonzero = enriched_tfidf.nnz

enriched_density = (
    enriched_nonzero
    / enriched_total_cells
)

enriched_sparsity = (
    1 - enriched_density
)


print("ENRICHED TF-IDF MATRIX")
print("=" * 80)

print(
    f"Documents:          "
    f"{enriched_tfidf.shape[0]:,}"
)

print(
    f"TF-IDF features:    "
    f"{enriched_tfidf.shape[1]:,}"
)

print(
    f"Non-zero values:    "
    f"{enriched_nonzero:,}"
)

print(
    f"Matrix density:     "
    f"{enriched_density:.6f}"
)

print(
    f"Matrix sparsity:    "
    f"{enriched_sparsity:.2%}"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert enriched_tfidf.shape[0] == 2067
assert enriched_tfidf.shape[1] > 0
assert enriched_tfidf.nnz > 0

print("\n✓ ENRICHED TF-IDF REPRESENTATION CREATED")

ENRICHED TF-IDF MATRIX
Documents:          2,067
TF-IDF features:    5,256
Non-zero values:    29,806
Matrix density:     0.002744
Matrix sparsity:    99.73%

✓ ENRICHED TF-IDF REPRESENTATION CREATED


In [15]:
# ============================================================
# CORE VS ENRICHED TF-IDF COMPARISON
# ============================================================

tfidf_comparison = pd.DataFrame({
    "representation": [
        "Core",
        "Enriched"
    ],

    "documents": [
        core_tfidf.shape[0],
        enriched_tfidf.shape[0]
    ],

    "features": [
        core_tfidf.shape[1],
        enriched_tfidf.shape[1]
    ],

    "nonzero_values": [
        core_tfidf.nnz,
        enriched_tfidf.nnz
    ],

    "density_pct": [
        core_density * 100,
        enriched_density * 100
    ],

    "sparsity_pct": [
        core_sparsity * 100,
        enriched_sparsity * 100
    ]
})


print("TF-IDF REPRESENTATION COMPARISON")
print("=" * 85)

display(
    tfidf_comparison.style.format({
        "density_pct": "{:.4f}%",
        "sparsity_pct": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# Vocabulary overlap
# ------------------------------------------------------------

core_vocab = set(core_feature_names)
enriched_vocab = set(enriched_feature_names)

shared_vocab = (
    core_vocab
    & enriched_vocab
)

core_only_vocab = (
    core_vocab
    - enriched_vocab
)

enriched_only_vocab = (
    enriched_vocab
    - core_vocab
)


print("\nVOCABULARY COMPARISON")
print("=" * 85)

print(f"Core vocabulary:          {len(core_vocab):,}")
print(f"Enriched vocabulary:      {len(enriched_vocab):,}")
print(f"Shared terms:             {len(shared_vocab):,}")
print(f"Core-only terms:          {len(core_only_vocab):,}")
print(f"Enriched-only terms:      {len(enriched_only_vocab):,}")

print(
    f"Core vocabulary retained: "
    f"{len(shared_vocab) / len(core_vocab):.2%}"
)

TF-IDF REPRESENTATION COMPARISON


,representation,documents,features,nonzero_values,density_pct,sparsity_pct
0,Core,2067,1935,9753,0.2438%,99.76%
1,Enriched,2067,5256,29806,0.2744%,99.73%



VOCABULARY COMPARISON
Core vocabulary:          1,935
Enriched vocabulary:      5,256
Shared terms:             1,935
Core-only terms:          0
Enriched-only terms:      3,321
Core vocabulary retained: 100.00%


## 6. Enriched Vocabulary Analysis

The Enriched TF-IDF representation contains substantially more features than the Core representation.

The Core vocabulary is fully retained, while the Enriched representation introduces additional terms derived primarily from subject metadata and descriptions.

However, additional vocabulary does not automatically imply a better recommendation representation. Because subject and description coverage differs substantially by source, the additional terms must be inspected for:

- domain relevance
- semantic specificity
- multilingual metadata
- bibliographic noise
- generic descriptive vocabulary
- source-specific vocabulary effects

This analysis therefore evaluates the additional vocabulary before selecting a representation for downstream similarity modelling.

In [16]:
# ============================================================
# ENRICHED VOCABULARY INSPECTION
# ============================================================

enriched_document_frequency = np.asarray(
    (enriched_tfidf > 0).sum(axis=0)
).ravel()


enriched_vocab_audit = (
    pd.DataFrame({
        "term": enriched_feature_names,
        "document_frequency": enriched_document_frequency
    })
    .sort_values(
        "document_frequency",
        ascending=False
    )
    .reset_index(drop=True)
)


print("MOST WIDESPREAD ENRICHED TF-IDF TERMS")
print("=" * 85)

display(
    enriched_vocab_audit.head(40)
)


# ------------------------------------------------------------
# Enriched-only vocabulary
# ------------------------------------------------------------

enriched_only_audit = (
    enriched_vocab_audit[
        enriched_vocab_audit["term"].isin(
            enriched_only_vocab
        )
    ]
    .sort_values(
        "document_frequency",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nMOST WIDESPREAD ENRICHED-ONLY TERMS")
print("=" * 85)

display(
    enriched_only_audit.head(40)
)

MOST WIDESPREAD ENRICHED TF-IDF TERMS


,term,document_frequency
0,management,578
1,leadership,359
2,business,308
3,organizational,183
4,economics,172
5,business economics,141
6,industrial,125
7,personnel,124
8,gestion,119
9,planning,116



MOST WIDESPREAD ENRICHED-ONLY TERMS


,term,document_frequency
0,business economics,141
1,industrial,125
2,personnel,124
3,gestion,119
4,personnel management,108
5,strategic planning,92
6,industrial management,89
7,case studies,73
8,organizational change,61
9,management management,61


In [17]:
# ============================================================
# SEMANTIC ENRICHMENT BY SOURCE
# ============================================================

semantic_source_audit = (
    books
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),

        books_with_subjects=(
            "has_subjects",
            "sum"
        ),

        books_with_description=(
            "has_description",
            "sum"
        ),

        median_core_words=(
            "core_word_count",
            "median"
        ),

        median_enriched_words=(
            "enriched_word_count",
            "median"
        ),

        mean_additional_words=(
            "additional_semantic_words",
            "mean"
        )
    )
    .reset_index()
)


semantic_source_audit[
    "subject_coverage_pct"
] = (
    semantic_source_audit["books_with_subjects"]
    / semantic_source_audit["books"]
    * 100
)


semantic_source_audit[
    "description_coverage_pct"
] = (
    semantic_source_audit["books_with_description"]
    / semantic_source_audit["books"]
    * 100
)


print("SEMANTIC ENRICHMENT BY SOURCE")
print("=" * 85)

display(
    semantic_source_audit.style.format({
        "median_core_words": "{:.1f}",
        "median_enriched_words": "{:.1f}",
        "mean_additional_words": "{:.2f}",
        "subject_coverage_pct": "{:.2f}%",
        "description_coverage_pct": "{:.2f}%"
    })
)

SEMANTIC ENRICHMENT BY SOURCE


,source_group,books,books_with_subjects,books_with_description,median_core_words,median_enriched_words,mean_additional_words,subject_coverage_pct,description_coverage_pct
0,Both,3,3,1,5.0,8.0,71.67,100.00%,33.33%
1,LeadershipNow only,1117,0,0,6.0,6.0,0.00,0.00%,0.00%
2,Open Library only,947,823,191,7.0,17.0,34.65,86.91%,20.17%


## 7. Book-Level TF-IDF Interpretation

TF-IDF representations are inspected at the individual-book level to determine whether high-weight features correspond to recognizable themes, authors, and subject concepts.

For selected books, the highest-weight terms are extracted from both the Core and Enriched representations.

This provides an interpretable diagnostic of how enrichment changes each book's numerical representation before similarity calculations are introduced.

In [18]:
# ============================================================
# BOOK-LEVEL TF-IDF INTERPRETATION
# ============================================================

def get_top_tfidf_terms(
    matrix,
    feature_names,
    row_index,
    top_n=12
):
    """
    Return the highest-weight TF-IDF terms
    for one document.
    """

    row = matrix.getrow(row_index)

    if row.nnz == 0:
        return pd.DataFrame(
            columns=["term", "tfidf_weight"]
        )

    order = np.argsort(
        row.data
    )[::-1][:top_n]

    return pd.DataFrame({
        "term": feature_names[
            row.indices[order]
        ],
        "tfidf_weight": row.data[order]
    })


print("✓ TF-IDF interpretation helper created")

✓ TF-IDF interpretation helper created


In [19]:
# ============================================================
# SELECT BOOKS FOR NLP INTERPRETATION
# ============================================================

# One LeadershipNow-only book
ln_example = (
    books[
        books["source_group"]
        == "LeadershipNow only"
    ]
    .sort_values("book_id")
    .iloc[0]
)


# One Open Library book with subjects + description
ol_example = (
    books[
        (books["source_group"] == "Open Library only")
        & (books["has_subjects"] == 1)
        & (books["has_description"] == 1)
    ]
    .sort_values(
        "enriched_word_count",
        ascending=False
    )
    .iloc[0]
)


# One cross-source book
both_example = (
    books[
        books["source_group"] == "Both"
    ]
    .sort_values("book_id")
    .iloc[0]
)


selected_ids = [
    ln_example["book_id"],
    ol_example["book_id"],
    both_example["book_id"]
]


selected_books = books[
    books["book_id"].isin(selected_ids)
][
    [
        "book_id",
        "canonical_title",
        "source_group",
        "content_depth",
        "core_word_count",
        "enriched_word_count"
    ]
]


print("SELECTED BOOKS")
print("=" * 90)

display(selected_books)

SELECTED BOOKS


,book_id,canonical_title,source_group,content_depth,core_word_count,enriched_word_count
15,BOOK00016,The leadership challenge,Both,Rich,13,220
166,BOOK00167,Leaders Eat Last,Open Library only,Rich,5,533
950,BOOK00951,Crisis Capable,LeadershipNow only,Basic,4,4


In [20]:
# ============================================================
# COMPARE CORE VS ENRICHED TOP TERMS
# ============================================================

for book_id in selected_ids:

    row_index = books.index[
        books["book_id"] == book_id
    ][0]

    title = books.loc[
        row_index,
        "canonical_title"
    ]

    source = books.loc[
        row_index,
        "source_group"
    ]


    print("\n" + "=" * 90)
    print(f"{book_id} | {title}")
    print(f"Source: {source}")
    print("=" * 90)


    print("\nCORE — TOP TF-IDF TERMS")

    display(
        get_top_tfidf_terms(
            core_tfidf,
            core_feature_names,
            row_index,
            top_n=12
        )
    )


    print("ENRICHED — TOP TF-IDF TERMS")

    display(
        get_top_tfidf_terms(
            enriched_tfidf,
            enriched_feature_names,
            row_index,
            top_n=12
        )
    )


BOOK00951 | Crisis Capable
Source: LeadershipNow only

CORE — TOP TF-IDF TERMS


,term,tfidf_weight
0,crisis,1.0


ENRICHED — TOP TF-IDF TERMS


,term,tfidf_weight
0,crisis,1.0



BOOK00167 | Leaders Eat Last
Source: Open Library only

CORE — TOP TF-IDF TERMS


...

,term,tfidf_weight
0,safety,0.173554
1,eat,0.152631
2,leaders,0.127202
3,trust,0.124552
4,deeply,0.123142
5,culture,0.121582
6,it's,0.120033
7,facing,0.118441
8,culture organizational,0.114794
9,line,0.114794



BOOK00016 | The leadership challenge
Source: Both

CORE — TOP TF-IDF TERMS


,term,tfidf_weight
0,posner,0.431635
1,barry posner,0.431635
2,barry,0.361444
3,leadership challenge,0.265049
4,challenge james,0.265049
5,kouzes barry,0.254931
6,elaine,0.254931
7,challenge,0.254931
8,kouzes,0.247082
9,james kouzes,0.247082


ENRICHED — TOP TF-IDF TERMS


,term,tfidf_weight
0,leadership challenge,0.206946
1,challenge,0.180011
2,barry,0.166678
3,kouzes barry,0.160589
4,barry posner,0.160589
5,posner,0.160589
6,kouzes,0.155645
7,changed,0.155645
8,books,0.152230
9,efficiency,0.140287


## 8. TF-IDF Coverage and Source Effects

Book-level inspection confirms that enriched metadata can introduce meaningful concepts not available from title and author alone. However, enrichment is not distributed uniformly across the catalogue.

To quantify this effect, TF-IDF activity is compared across source groups using:

- number of active TF-IDF features per book
- total TF-IDF weight
- Core versus Enriched feature expansion

This identifies whether the Enriched representation systematically provides richer vectors for one source group.

In [21]:
# ============================================================
# DOCUMENT-LEVEL TF-IDF ACTIVITY
# ============================================================

books["core_active_features"] = np.asarray(
    (core_tfidf > 0).sum(axis=1)
).ravel()

books["enriched_active_features"] = np.asarray(
    (enriched_tfidf > 0).sum(axis=1)
).ravel()


books["additional_active_features"] = (
    books["enriched_active_features"]
    - books["core_active_features"]
)


source_vector_summary = (
    books
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),

        median_core_features=(
            "core_active_features",
            "median"
        ),

        mean_core_features=(
            "core_active_features",
            "mean"
        ),

        median_enriched_features=(
            "enriched_active_features",
            "median"
        ),

        mean_enriched_features=(
            "enriched_active_features",
            "mean"
        ),

        median_added_features=(
            "additional_active_features",
            "median"
        ),

        mean_added_features=(
            "additional_active_features",
            "mean"
        )
    )
    .reset_index()
)


print("TF-IDF ACTIVITY BY SOURCE")
print("=" * 95)

display(
    source_vector_summary.style.format({
        "median_core_features": "{:.1f}",
        "mean_core_features": "{:.2f}",
        "median_enriched_features": "{:.1f}",
        "mean_enriched_features": "{:.2f}",
        "median_added_features": "{:.1f}",
        "mean_added_features": "{:.2f}"
    })
)

TF-IDF ACTIVITY BY SOURCE


,source_group,books,median_core_features,mean_core_features,median_enriched_features,mean_enriched_features,median_added_features,mean_added_features
0,Both,3,7.0,7.00,13.0,46.00,11.0,39.00
1,LeadershipNow only,1117,3.0,3.01,3.0,3.26,0.0,0.24
2,Open Library only,947,6.0,6.72,17.0,27.49,10.0,20.77


In [22]:
# ============================================================
# BOOK-LEVEL ENRICHMENT DISTRIBUTION
# ============================================================

books["tfidf_enrichment_ratio"] = np.where(
    books["core_active_features"] > 0,
    books["enriched_active_features"]
    / books["core_active_features"],
    np.nan
)


enrichment_summary = (
    books
    .groupby("source_group")[
        "tfidf_enrichment_ratio"
    ]
    .agg([
        "count",
        "median",
        "mean",
        "min",
        "max"
    ])
)


print("TF-IDF ENRICHMENT RATIO BY SOURCE")
print("=" * 95)

display(
    enrichment_summary.style.format({
        "median": "{:.2f}×",
        "mean": "{:.2f}×",
        "min": "{:.2f}×",
        "max": "{:.2f}×"
    })
)

TF-IDF ENRICHMENT RATIO BY SOURCE


,count,median,mean,min,max
source_group,,,,,
Both,3,6.50×,5.86×,1.57×,9.50×
LeadershipNow only,1076,1.00×,1.11×,1.00×,4.00×
Open Library only,946,2.60×,5.17×,1.00×,110.00×


## 9. Zero-Vector and Representation Coverage Audit

A non-empty text document does not necessarily produce a non-zero TF-IDF vector.

Terms may be excluded because they:

- occur in only one document and fail the `min_df=2` threshold,
- are removed as English stop words, or
- do not satisfy the tokenization rules.

Zero-vector documents are therefore identified explicitly because they cannot participate meaningfully in cosine-similarity calculations using that representation.

In [23]:
# ============================================================
# ZERO-VECTOR AUDIT
# ============================================================

books["core_zero_vector"] = (
    books["core_active_features"] == 0
)

books["enriched_zero_vector"] = (
    books["enriched_active_features"] == 0
)


print("ZERO-VECTOR AUDIT")
print("=" * 90)

print(
    f"Core zero-vector books:     "
    f"{books['core_zero_vector'].sum():,}"
)

print(
    f"Enriched zero-vector books: "
    f"{books['enriched_zero_vector'].sum():,}"
)


print("\nZERO VECTORS BY SOURCE")
print("=" * 90)

zero_by_source = (
    books
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),
        core_zero_vectors=("core_zero_vector", "sum"),
        enriched_zero_vectors=("enriched_zero_vector", "sum")
    )
    .reset_index()
)

display(zero_by_source)


print("\nCORE ZERO-VECTOR BOOKS")
print("=" * 90)

display(
    books.loc[
        books["core_zero_vector"],
        [
            "book_id",
            "canonical_title",
            "authors_text",
            "source_group",
            "core_content_text"
        ]
    ]
    .sort_values("book_id")
)

ZERO-VECTOR AUDIT
Core zero-vector books:     42
Enriched zero-vector books: 27

ZERO VECTORS BY SOURCE


,source_group,books,core_zero_vectors,enriched_zero_vectors
0,Both,3,0,0
1,LeadershipNow only,1117,41,27
2,Open Library only,947,1,0



CORE ZERO-VECTOR BOOKS


,book_id,canonical_title,authors_text,source_group,core_content_text
2,BOOK00003,Kepemimpinan =,Karjadi M.,Open Library only,Kepemimpinan = Karjadi M.
957,BOOK00958,The Kingdom of Prep,Maggie Bullock,LeadershipNow only,The Kingdom of Prep Maggie Bullock
962,BOOK00963,Next!,Joanne Lipman,LeadershipNow only,Next! Joanne Lipman
1001,BOOK01002,Unfiltered,Ana Lueneburger and Saurabh Mukherjea,LeadershipNow only,Unfiltered Ana Lueneburger and Saurabh Mukherjea
1029,BOOK01030,Nudging,Riccardo Viale,LeadershipNow only,Nudging Riccardo Viale
1059,BOOK01060,The Revolutionary,Stacy Schiff,LeadershipNow only,The Revolutionary Stacy Schiff
1061,BOOK01062,Get It Done,Ayelet Fishbach,LeadershipNow only,Get It Done Ayelet Fishbach
1071,BOOK01072,Enshittification,Cory Doctorow,LeadershipNow only,Enshittification Cory Doctorow
1117,BOOK01118,Unreasonable Hospitality,Will Guidara,LeadershipNow only,Unreasonable Hospitality Will Guidara
1131,BOOK01132,Million Dollar Weekend,Noah Kagan with Tahl Raz,LeadershipNow only,Million Dollar Weekend Noah Kagan with Tahl Raz


In [24]:
# ============================================================
# TF-IDF REPRESENTATION COVERAGE
# ============================================================

representation_coverage = (
    books
    .groupby("source_group")
    .agg(
        total_books=("book_id", "count"),
        core_nonzero_books=(
            "core_zero_vector",
            lambda x: (~x).sum()
        ),
        enriched_nonzero_books=(
            "enriched_zero_vector",
            lambda x: (~x).sum()
        )
    )
    .reset_index()
)


representation_coverage["core_coverage_pct"] = (
    representation_coverage["core_nonzero_books"]
    / representation_coverage["total_books"]
    * 100
)

representation_coverage["enriched_coverage_pct"] = (
    representation_coverage["enriched_nonzero_books"]
    / representation_coverage["total_books"]
    * 100
)


print("TF-IDF REPRESENTATION COVERAGE")
print("=" * 90)

display(
    representation_coverage.style.format({
        "core_coverage_pct": "{:.2f}%",
        "enriched_coverage_pct": "{:.2f}%"
    })
)

TF-IDF REPRESENTATION COVERAGE


,source_group,total_books,core_nonzero_books,enriched_nonzero_books,core_coverage_pct,enriched_coverage_pct
0,Both,3,3,3,100.00%,100.00%
1,LeadershipNow only,1117,1076,1090,96.33%,97.58%
2,Open Library only,947,946,947,99.89%,100.00%


## 10. Cross-Field Bigram Diagnostic

The Core representation concatenates title and author text into a single document.

This allows TF-IDF to capture useful title phrases and author names, but it can also create artificial bigrams at the boundary between a book title and its author, such as *challenge james*.

These boundary bigrams are quantified before finalizing the NLP representation. This diagnostic distinguishes legitimate semantic phrases from artifacts introduced by field concatenation.

In [25]:
# ============================================================
# CROSS-FIELD BIGRAM DIAGNOSTIC
# ============================================================

def last_valid_token(text):
    tokens = re.findall(
        r"\b[a-zA-Z][a-zA-Z'-]+\b",
        str(text).lower()
    )
    return tokens[-1] if tokens else None


def first_valid_token(text):
    tokens = re.findall(
        r"\b[a-zA-Z][a-zA-Z'-]+\b",
        str(text).lower()
    )
    return tokens[0] if tokens else None


books["title_last_token"] = (
    books["canonical_title"]
    .apply(last_valid_token)
)

books["author_first_token"] = (
    books["authors_text"]
    .apply(first_valid_token)
)


books["title_author_boundary_bigram"] = (
    books["title_last_token"].fillna("")
    + " "
    + books["author_first_token"].fillna("")
).str.strip()


core_bigram_vocab = {
    term
    for term in core_feature_names
    if " " in term
}


books["boundary_bigram_in_core_vocab"] = (
    books["title_author_boundary_bigram"]
    .isin(core_bigram_vocab)
)


boundary_count = (
    books["boundary_bigram_in_core_vocab"]
    .sum()
)

unique_boundary_bigrams = (
    books.loc[
        books["boundary_bigram_in_core_vocab"],
        "title_author_boundary_bigram"
    ]
    .nunique()
)


print("CROSS-FIELD BIGRAM DIAGNOSTIC")
print("=" * 90)

print(
    f"Books with title-author boundary bigram "
    f"in Core vocabulary: {boundary_count:,}"
)

print(
    f"Unique boundary bigrams represented: "
    f"{unique_boundary_bigrams:,}"
)

print(
    f"Share of catalogue affected: "
    f"{boundary_count / len(books):.2%}"
)


print("\nMOST COMMON TITLE-AUTHOR BOUNDARY BIGRAMS")
print("=" * 90)

display(
    books.loc[
        books["boundary_bigram_in_core_vocab"],
        "title_author_boundary_bigram"
    ]
    .value_counts()
    .head(25)
    .rename_axis("boundary_bigram")
    .reset_index(name="books")
)

CROSS-FIELD BIGRAM DIAGNOSTIC
Books with title-author boundary bigram in Core vocabulary: 391
Unique boundary bigrams represented: 127
Share of catalogue affected: 18.92%

MOST COMMON TITLE-AUTHOR BOUNDARY BIGRAMS


,boundary_bigram,books
0,management david,14
1,management michael,13
2,management gary,9
3,management robert,9
4,management john,9
5,leadership john,8
6,management jay,7
7,leadership david,6
8,management peter,6
9,management fred,6


## 11. Field-Boundary Correction

Diagnostic analysis identified artificial bigrams created when independently meaningful metadata fields were concatenated directly.

A title-author boundary bigram appeared in 391 books (18.92% of the catalogue), including combinations such as *management david* and *leadership john*.

These expressions are artifacts of text construction rather than meaningful semantic phrases.

To prevent cross-field n-grams while preserving the underlying metadata, explicit field boundaries are introduced before the final TF-IDF representations are fitted.

This correction is systematic and avoids subjective manual removal of individual vocabulary terms.

In [27]:
# ============================================================
# FIELD-BOUNDARY-AWARE NLP DOCUMENTS
# ============================================================

FIELD_BOUNDARY = "zzfieldboundaryzz"


def field_has_text(value):
    """
    Return True when a text field contains meaningful content.
    """
    
    if pd.isna(value):
        return False

    value = str(value).strip()

    return value.lower() not in {
        "",
        "[]",
        "{}",
        "nan",
        "none",
        "null",
        "na",
        "n/a"
    }


def join_nlp_fields(*fields):
    """
    Normalize independently meaningful text fields and join them
    using an explicit field-boundary marker.
    """

    cleaned_fields = [
        normalize_nlp_text(field)
        for field in fields
        if field_has_text(field)
    ]

    return (
        f" {FIELD_BOUNDARY} "
        .join(cleaned_fields)
    )


# ------------------------------------------------------------
# Core representation:
# title + author
# ------------------------------------------------------------

books["core_text_boundary"] = books.apply(
    lambda row: join_nlp_fields(
        row["canonical_title"],
        row["authors_text"]
    ),
    axis=1
)


# ------------------------------------------------------------
# Enriched representation:
# title + author + subjects + description
# ------------------------------------------------------------

books["enriched_text_boundary"] = books.apply(
    lambda row: join_nlp_fields(
        row["canonical_title"],
        row["authors_text"],
        row["subjects_text"],
        row["description_text"]
    ),
    axis=1
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(books) == 2067

assert (
    books["core_text_boundary"]
    .ne("")
    .all()
)

assert (
    books["enriched_text_boundary"]
    .ne("")
    .all()
)


print("FIELD-BOUNDARY DOCUMENTS CREATED")
print("=" * 90)

print(
    "Core documents:",
    books["core_text_boundary"].ne("").sum()
)

print(
    "Enriched documents:",
    books["enriched_text_boundary"].ne("").sum()
)


print("\nSAMPLE BOUNDARY-AWARE DOCUMENTS")
print("=" * 90)

display(
    books[
        [
            "book_id",
            "canonical_title",
            "core_text_boundary",
            "enriched_text_boundary"
        ]
    ].head(5)
)

FIELD-BOUNDARY DOCUMENTS CREATED
Core documents: 2067
Enriched documents: 2067

SAMPLE BOUNDARY-AWARE DOCUMENTS


,book_id,canonical_title,core_text_boundary,enriched_text_boundary
0,BOOK00001,Principle-Centered Leadership,principle-centered leadership zzfieldboundaryzz stephen r covey,principle-centered leadership zzfieldboundaryzz stephen r covey zzfieldboundaryzz leadership psychological aspects of success success psychologica...
1,BOOK00002,Leadership in Organizations,leadership in organizations zzfieldboundaryzz gary a yukl,leadership in organizations zzfieldboundaryzz gary a yukl zzfieldboundaryzz organisation prise de d cision entscheidungsfindung leadership organiz...
2,BOOK00003,Kepemimpinan =,kepemimpinan zzfieldboundaryzz karjadi m,kepemimpinan zzfieldboundaryzz karjadi m zzfieldboundaryzz leadership
3,BOOK00004,Spiritual leadership,spiritual leadership zzfieldboundaryzz j oswald sanders,spiritual leadership zzfieldboundaryzz j oswald sanders zzfieldboundaryzz christian leadership leadership spiritual directors
4,BOOK00005,Leadership,leadership zzfieldboundaryzz peter guy northouse,leadership zzfieldboundaryzz peter guy northouse zzfieldboundaryzz cas tudes de leiderschap leadership case studies f hrung leadership--case studi...


## 12. Boundary-Aware TF-IDF Tokenization

To eliminate artificial n-grams across metadata fields, the final TF-IDF representation uses a custom analyzer.

Each document is divided at the explicit field boundary. Unigrams and bigrams are then generated independently within each field.

This preserves meaningful phrases such as:

- project management
- emotional intelligence
- organizational behavior
- transformational leadership

while preventing artificial combinations such as:

- management david
- leadership john
- challenge james

The same analyzer is applied to both Core and Enriched representations.

In [29]:
# ============================================================
# BOUNDARY-AWARE ANALYZER
# ============================================================

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


STOP_WORDS = set(ENGLISH_STOP_WORDS)


def boundary_aware_analyzer(document):
    """
    Generate unigrams and bigrams independently within each
    metadata field.

    Cross-field bigrams are impossible because each field is
    tokenized separately.
    """

    features = []

    # Split document back into independent metadata fields
    fields = document.split(FIELD_BOUNDARY)

    for field in fields:

        # Extract alphabetic tokens of length >= 2
        tokens = re.findall(
            r"\b[a-zA-Z][a-zA-Z'-]+\b",
            field.lower()
        )

        # Standard English stop-word removal
        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        # Unigrams
        features.extend(tokens)

        # Bigrams — only inside the current field
        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(len(tokens) - 1)
            ]
        )

    return features


# ------------------------------------------------------------
# Quick validation
# ------------------------------------------------------------

test_document = (
    "strategic management "
    + FIELD_BOUNDARY +
    " michael armstrong"
)

test_features = boundary_aware_analyzer(
    test_document
)


print("BOUNDARY-AWARE ANALYZER TEST")
print("=" * 80)

print(test_features)


assert "strategic management" in test_features
assert "michael armstrong" in test_features

# This artificial title-author bigram must NOT exist
assert "management michael" not in test_features

print("\n✓ Cross-field bigram successfully prevented")

BOUNDARY-AWARE ANALYZER TEST
['strategic', 'management', 'strategic management', 'michael', 'armstrong', 'michael armstrong']

✓ Cross-field bigram successfully prevented


In [30]:
# ============================================================
# FINAL BOUNDARY-AWARE TF-IDF REPRESENTATIONS
# ============================================================

# ------------------------------------------------------------
# Core TF-IDF
# ------------------------------------------------------------

core_vectorizer_final = TfidfVectorizer(
    analyzer=boundary_aware_analyzer,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

core_tfidf_final = core_vectorizer_final.fit_transform(
    books["core_text_boundary"]
)

core_feature_names_final = (
    core_vectorizer_final.get_feature_names_out()
)


# ------------------------------------------------------------
# Enriched TF-IDF
# ------------------------------------------------------------

enriched_vectorizer_final = TfidfVectorizer(
    analyzer=boundary_aware_analyzer,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

enriched_tfidf_final = (
    enriched_vectorizer_final.fit_transform(
        books["enriched_text_boundary"]
    )
)

enriched_feature_names_final = (
    enriched_vectorizer_final.get_feature_names_out()
)


# ------------------------------------------------------------
# Matrix diagnostics
# ------------------------------------------------------------

def tfidf_diagnostics(name, matrix):
    
    rows, cols = matrix.shape
    total_cells = rows * cols
    density = matrix.nnz / total_cells
    sparsity = 1 - density

    return {
        "representation": name,
        "documents": rows,
        "features": cols,
        "nonzero_values": matrix.nnz,
        "density_pct": density * 100,
        "sparsity_pct": sparsity * 100
    }


tfidf_final_summary = pd.DataFrame([
    tfidf_diagnostics(
        "Core",
        core_tfidf_final
    ),
    tfidf_diagnostics(
        "Enriched",
        enriched_tfidf_final
    )
])


print("FINAL BOUNDARY-AWARE TF-IDF MATRICES")
print("=" * 90)

display(
    tfidf_final_summary.style.format({
        "density_pct": "{:.4f}%",
        "sparsity_pct": "{:.2f}%"
    })
)

FINAL BOUNDARY-AWARE TF-IDF MATRICES


,representation,documents,features,nonzero_values,density_pct,sparsity_pct
0,Core,2067,1810,9364,0.2503%,99.75%
1,Enriched,2067,5065,29245,0.2793%,99.72%


In [31]:
# ============================================================
# VERIFY CROSS-FIELD BIGRAM REMOVAL
# ============================================================

previous_boundary_terms = set(
    books.loc[
        books["boundary_bigram_in_core_vocab"],
        "title_author_boundary_bigram"
    ]
)


final_core_vocabulary = set(
    core_feature_names_final
)


remaining_boundary_terms = (
    previous_boundary_terms
    & final_core_vocabulary
)


print("BOUNDARY ARTIFACT VALIDATION")
print("=" * 90)

print(
    f"Previously identified boundary terms: "
    f"{len(previous_boundary_terms):,}"
)

print(
    f"Remaining in final Core vocabulary: "
    f"{len(remaining_boundary_terms):,}"
)


if remaining_boundary_terms:
    print("\nTerms still present:")
    
    print(
        sorted(remaining_boundary_terms)[:30]
    )

else:
    print(
        "\n✓ All previously identified "
        "title-author boundary artifacts removed."
    )

BOUNDARY ARTIFACT VALIDATION
Previously identified boundary terms: 127
Remaining in final Core vocabulary: 2

Terms still present:
['guide project', 'king john']


In [32]:
# ============================================================
# UNICODE / MULTILINGUAL TEXT AUDIT
# ============================================================

unicode_mask = (
    books["enriched_content_text"]
    .fillna("")
    .str.contains(r"[^\x00-\x7F]", regex=True)
)

unicode_books = books.loc[
    unicode_mask,
    [
        "book_id",
        "canonical_title",
        "source_group",
        "enriched_content_text",
        "enriched_text_clean"
    ]
].copy()


print("UNICODE / MULTILINGUAL TEXT AUDIT")
print("=" * 90)

print(
    f"Books containing non-ASCII characters: "
    f"{unicode_mask.sum():,}"
)

print(
    f"Share of catalogue: "
    f"{unicode_mask.mean():.2%}"
)


print("\nBY SOURCE")
print("=" * 90)

display(
    books.loc[unicode_mask]
    .groupby("source_group")
    .size()
    .rename("books_with_non_ascii")
    .reset_index()
)


print("\nSAMPLE RECORDS")
print("=" * 90)

display(
    unicode_books.head(15)
)

UNICODE / MULTILINGUAL TEXT AUDIT
Books containing non-ASCII characters: 225
Share of catalogue: 10.89%

BY SOURCE


,source_group,books_with_non_ascii
0,Both,1
1,LeadershipNow only,23
2,Open Library only,201



SAMPLE RECORDS


,book_id,canonical_title,source_group,enriched_content_text,enriched_text_clean
1,BOOK00002,Leadership in Organizations,Open Library only,Leadership in Organizations Gary A. Yukl Organisation Prise de décision Entscheidungsfindung Leadership Organizational sociology Organization Deci...,leadership in organizations gary a yukl organisation prise de d cision entscheidungsfindung leadership organizational sociology organization decis...
4,BOOK00005,Leadership,Open Library only,"Leadership Peter Guy Northouse Cas, Études de Leiderschap Leadership Case studies Führung Leadership--case studies Hm141 .n67 1997 303.3/4",leadership peter guy northouse cas tudes de leiderschap leadership case studies f hrung leadership--case studies hm141 n67 1997 303 3 4
5,BOOK00006,The 21 Irrefutable Laws of Leadership,Open Library only,The 21 Irrefutable Laws of Leadership John C. Maxwell Leadership Industrial management Dirección y administración Liderazgo Industria Gestion d'en...,the 21 irrefutable laws of leadership john c maxwell leadership industrial management direcci n y administraci n liderazgo industria gestion d'ent...
6,BOOK00007,Leadership,Open Library only,"Leadership Peter G. Northouse Leadership Case studies Sociology Führung Leadership--case studies Hm1261 .n67 2018 303.3/4 ""The Third Edition of th...",leadership peter g northouse leadership case studies sociology f hrung leadership--case studies hm1261 n67 2018 303 3 4 the third edition of this ...
9,BOOK00010,Leadership and Self Deception,Open Library only,Leadership and Self Deception The Arbinger Institute Dick Ruhe Berrett Koehler Business Nonfiction Leadership Self-deception Déception de soi BUSI...,leadership and self deception the arbinger institute dick ruhe berrett koehler business nonfiction leadership self-deception d ception de soi busi...
12,BOOK00013,Lincoln on Leadership,Open Library only,Lincoln on Leadership Donald T. Phillips Views on political leadership Political leadership Leadership Politische Führung Leiderschap Politische F...,lincoln on leadership donald t phillips views on political leadership political leadership leadership politische f hrung leiderschap politische fu...
13,BOOK00014,Leadership,Open Library only,Leadership James MacGregor Burns Leadership Fuhrungspsychologie Fuhrungseigenschaft Politische Fuhrung Politieke aspecten Führung Pouvoir (scie...,leadership james macgregor burns leadership fu hrungspsychologie fu hrungseigenschaft politische fu hrung politieke aspecten f hrung pouvoir scien...
15,BOOK00016,The leadership challenge,Both,The leadership challenge James M. Kouzes Barry Z. Posner Elaine Biech Barry Posner Executive ability Leadership Management Business Nonfiction Lei...,the leadership challenge james m kouzes barry z posner elaine biech barry posner executive ability leadership management business nonfiction leide...
18,BOOK00019,Leadership In Turbulent Times,Open Library only,Leadership In Turbulent Times Doris Kearns Goodwin Doris Kearns Goodwin Politics and government Political culture Political leadership Presidents ...,leadership in turbulent times doris kearns goodwin doris kearns goodwin politics and government political culture political leadership presidents ...
19,BOOK00020,Leadership,Open Library only,Leadership Michael Z. Hackman Craig E. Johnson Communication Leadership Politics - Current Events Business & Economics / Management Science Politi...,leadership michael z hackman craig e johnson communication leadership politics - current events business economics management science politics cur...


In [33]:
# ============================================================
# UNICODE-SAFE NLP NORMALIZATION
# ============================================================

def normalize_nlp_text(text):
    """
    Conservative Unicode-safe normalization.

    - lowercase
    - standardize apostrophes
    - preserve Unicode letters and numbers
    - preserve apostrophes and hyphens
    - remove other punctuation
    - normalize whitespace

    No stemming, lemmatization, translation, or accent removal.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Standardize curly apostrophes
    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
    )

    # Preserve Unicode word characters, apostrophes,
    # hyphens and whitespace
    text = re.sub(
        r"[^\w'\-\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Treat underscore as separator
    text = text.replace("_", " ")

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


print("UNICODE NORMALIZATION TEST")
print("=" * 90)

test_terms = [
    "Prise de décision",
    "Dirección y administración",
    "Führung",
    "Déception de soi",
    "Taoísmo",
    "d'entreprise"
]

for term in test_terms:
    print(
        f"{term:<35} -> "
        f"{normalize_nlp_text(term)}"
    )

UNICODE NORMALIZATION TEST
Prise de décision                   -> prise de décision
Dirección y administración          -> dirección y administración
Führung                             -> führung
Déception de soi                    -> déception de soi
Taoísmo                             -> taoísmo
d'entreprise                        -> d'entreprise


In [34]:
# ============================================================
# UNICODE-SAFE BOUNDARY-AWARE ANALYZER
# ============================================================

def boundary_aware_analyzer(document):
    """
    Generate Unicode-aware unigrams and bigrams independently
    within each metadata field.

    Cross-field bigrams are prevented by processing each field
    separately.
    """

    features = []

    fields = document.split(FIELD_BOUNDARY)

    for field in fields:

        # Unicode-aware tokens, minimum length of 2 characters
        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        # Remove standard English stop words only
        # Multilingual vocabulary is deliberately retained.
        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        # Unigrams
        features.extend(tokens)

        # Bigrams within the SAME field only
        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(len(tokens) - 1)
            ]
        )

    return features

In [35]:
# ============================================================
# UNICODE + FIELD-BOUNDARY VALIDATION
# ============================================================

test_document = (
    "Dirección y administración "
    + FIELD_BOUNDARY +
    " José García"
)

test_features = boundary_aware_analyzer(
    test_document
)

print("UNICODE-AWARE ANALYZER TEST")
print("=" * 90)

print(test_features)


# Unicode should survive
assert "dirección" in test_features
assert "administración" in test_features
assert "josé" in test_features
assert "garcía" in test_features

# Legitimate within-field bigram
assert "josé garcía" in test_features

# Artificial cross-field bigram must not exist
assert "administración josé" not in test_features


print(
    "\n✓ Unicode preserved"
)

print(
    "✓ Within-field bigrams preserved"
)

print(
    "✓ Cross-field bigrams prevented"
)

UNICODE-AWARE ANALYZER TEST
['dirección', 'administración', 'dirección administración', 'josé', 'garcía', 'josé garcía']

✓ Unicode preserved
✓ Within-field bigrams preserved
✓ Cross-field bigrams prevented


In [36]:
# ============================================================
# REGENERATE UNICODE-SAFE BOUNDARY DOCUMENTS
# ============================================================

books["core_text_boundary"] = books.apply(
    lambda row: join_nlp_fields(
        row["canonical_title"],
        row["authors_text"]
    ),
    axis=1
)

books["enriched_text_boundary"] = books.apply(
    lambda row: join_nlp_fields(
        row["canonical_title"],
        row["authors_text"],
        row["subjects_text"],
        row["description_text"]
    ),
    axis=1
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(books) == 2067
assert books["core_text_boundary"].ne("").all()
assert books["enriched_text_boundary"].ne("").all()


print("UNICODE-SAFE BOUNDARY DOCUMENTS")
print("=" * 90)

print(
    f"Core documents:     "
    f"{books['core_text_boundary'].ne('').sum():,}"
)

print(
    f"Enriched documents: "
    f"{books['enriched_text_boundary'].ne('').sum():,}"
)


# ------------------------------------------------------------
# Inspect records known to contain multilingual metadata
# ------------------------------------------------------------

sample_ids = [
    "BOOK00002",
    "BOOK00005",
    "BOOK00006",
    "BOOK00010",
    "BOOK00028"
]

display(
    books.loc[
        books["book_id"].isin(sample_ids),
        [
            "book_id",
            "canonical_title",
            "enriched_text_boundary"
        ]
    ]
)

UNICODE-SAFE BOUNDARY DOCUMENTS
Core documents:     2,067
Enriched documents: 2,067


,book_id,canonical_title,enriched_text_boundary
1,BOOK00002,Leadership in Organizations,leadership in organizations zzfieldboundaryzz gary a yukl zzfieldboundaryzz organisation prise de décision entscheidungsfindung leadership organiz...
4,BOOK00005,Leadership,leadership zzfieldboundaryzz peter guy northouse zzfieldboundaryzz cas études de leiderschap leadership case studies führung leadership--case stud...
5,BOOK00006,The 21 Irrefutable Laws of Leadership,the 21 irrefutable laws of leadership zzfieldboundaryzz john c maxwell zzfieldboundaryzz leadership industrial management dirección y administraci...
9,BOOK00010,Leadership and Self Deception,leadership and self deception zzfieldboundaryzz the arbinger institute dick ruhe zzfieldboundaryzz berrett koehler business nonfiction leadership ...
27,BOOK00028,The Tao of leadership,the tao of leadership zzfieldboundaryzz john heider zzfieldboundaryzz taoísmo psychological aspects of leadership leadership psychological aspects...


In [37]:
# ============================================================
# FINAL UNICODE-SAFE BOUNDARY-AWARE TF-IDF
# ============================================================

core_vectorizer_final = TfidfVectorizer(
    analyzer=boundary_aware_analyzer,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

core_tfidf_final = core_vectorizer_final.fit_transform(
    books["core_text_boundary"]
)

core_feature_names_final = (
    core_vectorizer_final.get_feature_names_out()
)


enriched_vectorizer_final = TfidfVectorizer(
    analyzer=boundary_aware_analyzer,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

enriched_tfidf_final = enriched_vectorizer_final.fit_transform(
    books["enriched_text_boundary"]
)

enriched_feature_names_final = (
    enriched_vectorizer_final.get_feature_names_out()
)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

def final_tfidf_diagnostics(name, matrix):
    
    rows, cols = matrix.shape
    density = matrix.nnz / (rows * cols)

    return {
        "representation": name,
        "documents": rows,
        "features": cols,
        "nonzero_values": matrix.nnz,
        "density_pct": density * 100,
        "sparsity_pct": (1 - density) * 100
    }


final_tfidf_summary = pd.DataFrame([
    final_tfidf_diagnostics(
        "Core",
        core_tfidf_final
    ),
    final_tfidf_diagnostics(
        "Enriched",
        enriched_tfidf_final
    )
])


print("FINAL UNICODE-SAFE TF-IDF MATRICES")
print("=" * 90)

display(
    final_tfidf_summary.style.format({
        "density_pct": "{:.4f}%",
        "sparsity_pct": "{:.2f}%"
    })
)

FINAL UNICODE-SAFE TF-IDF MATRICES


,representation,documents,features,nonzero_values,density_pct,sparsity_pct
0,Core,2067,1816,9372,0.2497%,99.75%
1,Enriched,2067,5130,29372,0.2770%,99.72%


In [38]:
# ============================================================
# FINAL ZERO-VECTOR AUDIT
# ============================================================

books["core_active_features_final"] = (
    np.asarray(
        (core_tfidf_final != 0).sum(axis=1)
    )
    .ravel()
)

books["enriched_active_features_final"] = (
    np.asarray(
        (enriched_tfidf_final != 0).sum(axis=1)
    )
    .ravel()
)


books["core_zero_vector_final"] = (
    books["core_active_features_final"] == 0
)

books["enriched_zero_vector_final"] = (
    books["enriched_active_features_final"] == 0
)


final_coverage = (
    books
    .groupby("source_group")
    .agg(
        total_books=("book_id", "count"),
        core_zero_vectors=(
            "core_zero_vector_final",
            "sum"
        ),
        enriched_zero_vectors=(
            "enriched_zero_vector_final",
            "sum"
        )
    )
    .reset_index()
)


final_coverage["core_coverage_pct"] = (
    (
        final_coverage["total_books"]
        - final_coverage["core_zero_vectors"]
    )
    / final_coverage["total_books"]
    * 100
)


final_coverage["enriched_coverage_pct"] = (
    (
        final_coverage["total_books"]
        - final_coverage["enriched_zero_vectors"]
    )
    / final_coverage["total_books"]
    * 100
)


print("FINAL TF-IDF REPRESENTATION COVERAGE")
print("=" * 90)

print(
    f"Core zero vectors: "
    f"{books['core_zero_vector_final'].sum():,}"
)

print(
    f"Enriched zero vectors: "
    f"{books['enriched_zero_vector_final'].sum():,}"
)


display(
    final_coverage.style.format({
        "core_coverage_pct": "{:.2f}%",
        "enriched_coverage_pct": "{:.2f}%"
    })
)

FINAL TF-IDF REPRESENTATION COVERAGE
Core zero vectors: 42
Enriched zero vectors: 27


,source_group,total_books,core_zero_vectors,enriched_zero_vectors,core_coverage_pct,enriched_coverage_pct
0,Both,3,0,0,100.00%,100.00%
1,LeadershipNow only,1117,41,27,96.33%,97.58%
2,Open Library only,947,1,0,99.89%,100.00%


## 13. Final NLP Representation

The final NLP pipeline uses Unicode-safe, field-boundary-aware TF-IDF representations.

Two representations are retained:

- **Core TF-IDF:** title and author metadata.
- **Enriched TF-IDF:** title, author, subjects, and description.

The final preprocessing strategy:

- preserves Unicode and multilingual metadata;
- applies conservative lowercasing and punctuation normalization;
- removes standard English stop words;
- retains domain-specific terminology;
- generates unigrams and bigrams within metadata fields;
- prevents artificial n-grams across field boundaries;
- uses `min_df=2` to exclude corpus-unique terms;
- uses `max_df=0.95`;
- applies sublinear term-frequency scaling.

The Core matrix provides the more source-comparable representation, while the Enriched matrix captures additional semantic information where subjects and descriptions are available.

Books producing zero TF-IDF vectors are retained in the canonical dataset and explicitly flagged rather than removed or artificially imputed.

In [39]:
# ============================================================
# SAVE FINAL NLP ARTIFACTS
# ============================================================

import joblib
from scipy import sparse


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

CORE_MATRIX_PATH = (
    MODELS_DIR / "core_tfidf_matrix.npz"
)

ENRICHED_MATRIX_PATH = (
    MODELS_DIR / "enriched_tfidf_matrix.npz"
)

CORE_VECTORIZER_PATH = (
    MODELS_DIR / "core_tfidf_vectorizer.joblib"
)

ENRICHED_VECTORIZER_PATH = (
    MODELS_DIR / "enriched_tfidf_vectorizer.joblib"
)

NLP_INDEX_PATH = (
    DATA_PROCESSED / "nlp_book_index.csv"
)

NLP_FEATURES_PATH = (
    DATA_PROCESSED / "books_nlp_features.csv"
)

NLP_SUMMARY_PATH = (
    DATA_PROCESSED / "nlp_summary.csv"
)


# ------------------------------------------------------------
# Save sparse matrices
# ------------------------------------------------------------

sparse.save_npz(
    CORE_MATRIX_PATH,
    core_tfidf_final
)

sparse.save_npz(
    ENRICHED_MATRIX_PATH,
    enriched_tfidf_final
)


# ------------------------------------------------------------
# Save fitted vectorizers
# ------------------------------------------------------------

joblib.dump(
    core_vectorizer_final,
    CORE_VECTORIZER_PATH
)

joblib.dump(
    enriched_vectorizer_final,
    ENRICHED_VECTORIZER_PATH
)


# ------------------------------------------------------------
# Matrix row → book mapping
# ------------------------------------------------------------

nlp_book_index = pd.DataFrame({
    "matrix_row": np.arange(len(books)),
    "book_id": books["book_id"],
    "canonical_title": books["canonical_title"],
    "source_group": books["source_group"],
    "core_zero_vector": books["core_zero_vector_final"],
    "enriched_zero_vector": books["enriched_zero_vector_final"]
})


nlp_book_index.to_csv(
    NLP_INDEX_PATH,
    index=False
)


# ------------------------------------------------------------
# NLP-derived book features
# ------------------------------------------------------------

nlp_features = books[
    [
        "book_id",
        "canonical_title",
        "source_group",
        "core_text_boundary",
        "enriched_text_boundary",
        "core_active_features_final",
        "enriched_active_features_final",
        "core_zero_vector_final",
        "enriched_zero_vector_final"
    ]
].copy()


nlp_features.to_csv(
    NLP_FEATURES_PATH,
    index=False
)


# ------------------------------------------------------------
# NLP summary
# ------------------------------------------------------------

nlp_summary = pd.DataFrame({
    "metric": [
        "books",
        "core_features",
        "enriched_features",
        "core_nonzero_values",
        "enriched_nonzero_values",
        "core_sparsity_pct",
        "enriched_sparsity_pct",
        "core_zero_vectors",
        "enriched_zero_vectors"
    ],

    "value": [
        len(books),
        core_tfidf_final.shape[1],
        enriched_tfidf_final.shape[1],
        core_tfidf_final.nnz,
        enriched_tfidf_final.nnz,
        (
            1
            - core_tfidf_final.nnz
            / (
                core_tfidf_final.shape[0]
                * core_tfidf_final.shape[1]
            )
        ) * 100,
        (
            1
            - enriched_tfidf_final.nnz
            / (
                enriched_tfidf_final.shape[0]
                * enriched_tfidf_final.shape[1]
            )
        ) * 100,
        books["core_zero_vector_final"].sum(),
        books["enriched_zero_vector_final"].sum()
    ]
})


nlp_summary.to_csv(
    NLP_SUMMARY_PATH,
    index=False
)


print("FINAL NLP ARTIFACTS SAVED")
print("=" * 90)

for path in [
    CORE_MATRIX_PATH,
    ENRICHED_MATRIX_PATH,
    CORE_VECTORIZER_PATH,
    ENRICHED_VECTORIZER_PATH,
    NLP_INDEX_PATH,
    NLP_FEATURES_PATH,
    NLP_SUMMARY_PATH
]:
    print(path)

FINAL NLP ARTIFACTS SAVED
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/core_tfidf_matrix.npz
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_matrix.npz
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/core_tfidf_vectorizer.joblib
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_vectorizer.joblib
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/nlp_book_index.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/books_nlp_features.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/nlp_summary.csv


In [40]:
# ============================================================
# RELOAD AND VALIDATE SAVED NLP ARTIFACTS
# ============================================================

core_matrix_check = sparse.load_npz(
    CORE_MATRIX_PATH
)

enriched_matrix_check = sparse.load_npz(
    ENRICHED_MATRIX_PATH
)

core_vectorizer_check = joblib.load(
    CORE_VECTORIZER_PATH
)

enriched_vectorizer_check = joblib.load(
    ENRICHED_VECTORIZER_PATH
)

nlp_index_check = pd.read_csv(
    NLP_INDEX_PATH
)

nlp_features_check = pd.read_csv(
    NLP_FEATURES_PATH
)

nlp_summary_check = pd.read_csv(
    NLP_SUMMARY_PATH
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert core_matrix_check.shape == (
    2067,
    1816
)

assert enriched_matrix_check.shape == (
    2067,
    5130
)

assert len(
    core_vectorizer_check.get_feature_names_out()
) == 1816

assert len(
    enriched_vectorizer_check.get_feature_names_out()
) == 5130

assert len(nlp_index_check) == 2067

assert (
    nlp_index_check["book_id"].nunique()
    == 2067
)

assert len(nlp_features_check) == 2067

assert (
    nlp_features_check["book_id"].nunique()
    == 2067
)


# Matrix row mapping must remain exact
assert (
    nlp_index_check
    .sort_values("matrix_row")["book_id"]
    .tolist()
    ==
    books["book_id"].tolist()
)


print("NLP ARTIFACT VALIDATION")
print("=" * 90)

print(
    "Core matrix:",
    core_matrix_check.shape
)

print(
    "Enriched matrix:",
    enriched_matrix_check.shape
)

print(
    "Core vocabulary:",
    len(
        core_vectorizer_check
        .get_feature_names_out()
    )
)

print(
    "Enriched vocabulary:",
    len(
        enriched_vectorizer_check
        .get_feature_names_out()
    )
)

print(
    "Book index:",
    len(nlp_index_check)
)

print(
    "NLP feature records:",
    len(nlp_features_check)
)

print("\n✓ All saved NLP artifacts reloaded successfully.")
print("✓ Matrix-to-book mapping validated.")

NLP ARTIFACT VALIDATION
Core matrix: (2067, 1816)
Enriched matrix: (2067, 5130)
Core vocabulary: 1816
Enriched vocabulary: 5130
Book index: 2067
NLP feature records: 2067

✓ All saved NLP artifacts reloaded successfully.
✓ Matrix-to-book mapping validated.


## 14. NLP Conclusions

The NLP stage transformed textual metadata from 2,067 leadership and management books into numerical representations suitable for unsupervised machine learning and content-based recommendation.

### Final TF-IDF Representations

Two complementary representations were retained:

- **Core TF-IDF:** title and author metadata
  - 2,067 books
  - 1,816 features
  - 9,372 non-zero values
  - approximately 99.75% sparse
  - 42 zero-vector books

- **Enriched TF-IDF:** title, author, subjects, and description
  - 2,067 books
  - 5,130 features
  - 29,372 non-zero values
  - approximately 99.72% sparse
  - 27 zero-vector books

### Methodological Findings

The Core representation provides greater comparability across data sources because title and author information are available for nearly the entire catalogue.

The Enriched representation contains substantially more semantic information, but its additional metadata is unevenly distributed across sources. Open Library records are considerably more likely to contain subjects and descriptions than LeadershipNow records. Therefore, greater textual richness should not automatically be interpreted as superior model quality.

A field-boundary diagnostic identified artificial bigrams caused by directly concatenating independent metadata fields. The final NLP pipeline therefore generates unigrams and bigrams separately within each field, preventing artificial title-author combinations while preserving meaningful phrases such as *project management*, *emotional intelligence*, and *organizational behavior*.

Unicode-safe preprocessing was also implemented after identifying multilingual metadata in the corpus. Accented and non-English terms are preserved rather than translated, stripped, or artificially standardized.

Standard English stop words are removed, while leadership and management terminology is deliberately retained because these terms carry domain meaning.

The final TF-IDF matrices remain highly sparse, which is expected for document-term representations. This sparsity supports the use of sparse-compatible dimensionality-reduction techniques in the subsequent modelling stage.

Books producing zero TF-IDF vectors are retained in the canonical catalogue and explicitly flagged. They are not removed or assigned fabricated textual features.

### Modelling Implication

The finalized TF-IDF matrices provide the numerical text representations required for the unsupervised machine-learning pipeline.

The next stage will examine dimensionality reduction before clustering. Because TF-IDF is a high-dimensional sparse representation, Truncated Singular Value Decomposition (Truncated SVD / Latent Semantic Analysis) will be evaluated for text reduction rather than applying standard PCA directly to the sparse matrices.

The reduced semantic representations can subsequently be used for clustering, cluster interpretation, visualization, and the content-based recommendation system.